<a href="https://colab.research.google.com/github/VV-41/test/blob/test/YOLOMODEL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
parserpixy_ip102_dataset_path = kagglehub.dataset_download('parserpixy/ip102-dataset')

print('Data source import complete.')


100%|██████████| 770M/770M [00:04<00:00, 165MB/s]

Extracting files...


Data source import complete.


In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!find /kaggle/input -maxdepth 4 -type d

/kaggle/input


In [5]:
!pip install -U torch torchvision
!pip install -U torch_xla[tpu]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.6/554.6 MB 776.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 553.1/553.1 MB 885.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.0/216.0 MB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 24.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 875.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 156.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 109.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.1/214.1 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 77.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 MB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2

INFO: pip is looking at multiple versions of torch-xla[tpu] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.9/106.9 MB 5.0 MB/s eta 0:00:00
  Attempting uninstall: libtpu
    Found existing installation: libtpu 0.0.21.1
    Uninstalling libtpu-0.0.21.1:
      Successfully uninstalled libtpu-0.0.21.1
^C


In [4]:
import torch
print(torch.__version__, torch.version.cuda)
print(torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))
x = torch.randn(3, 3, device="cuda")
print(x @ x)

2.9.0+cpu None


AssertionError: Torch not compiled with CUDA enabled

In [ ]:
"""
=============================================================================
IP102 Object Detection — Research-Ready Training Pipeline
Ultralytics YOLOv8 / YOLOv9 / YOLOv10 / YOLOv11
=============================================================================

WHAT CHANGED FROM THE BASELINE SCRIPT, AND WHY
-----------------------------------------------
This is an enhanced version of a straightforward VOC->YOLO conversion +
train script. The changes below are specifically informed by two things
observed in this project so far:

1. Your classification run (YOLOv8s-cls) silently lost one class during
   parsing — 102 classes were loaded, but only 101 ended up in the final
   train/val/test splits, with 669/111/335 unparseable lines in the split
   files. That failure was easy to miss in the logs. This version makes
   that class of bug IMPOSSIBLE to miss: dataset integrity is validated
   up front and the run stops (or is explicitly overridden) if class
   counts don't match, and every dropped line/image is logged with a
   reason.

2. Literature review (Swin-AARNet, RepMSNet, Pest-ConFormer, DWViT-ES)
   showed the field plateaus around 76-79% top-1 on the classification
   side largely BECAUSE of two compounding issues: severe long-tailed
   class imbalance (IP102's imbalance ratio exceeds 91:1 at the
   super-class level) and fine-grained inter-class similarity. This
   detection script directly addresses the imbalance half of that
   problem — the fine-grained-similarity half is architectural (out of
   scope for a detector's data pipeline) but is noted in comments where
   relevant.

3. NEW: A run on a Tesla P100 hit `AcceleratorError: CUDA error: no
   kernel image is available for execution on the device` deep inside
   ultralytics' trainer, at the `model.to(device)` call. Root cause:
   torch.cuda.is_available() returns True and device discovery succeeds,
   but the installed PyTorch wheel simply has no compiled CUDA kernels
   for the P100's compute capability (sm_60) — recent PyTorch CUDA
   wheels (cu128 and newer indices) dropped sm_50/sm_60 support in favor
   of sm_100/sm_120 (Blackwell). This is a wheel/GPU mismatch, not a bug
   in this script's logic — but the OLD device check here
   (`torch.cuda.is_available()`) can't detect it, since availability and
   kernel-executability are different things. This version adds a real
   preflight GPU op (see `check_cuda_kernel_compatibility`) that catches
   this BEFORE the model/training even loads, and fails loudly with the
   exact fix instead of surfacing a 40-line stack trace mid-training.

   THE ACTUAL FIX (must be done before running this script, since torch
   can't be hot-swapped once imported in a running process):
       pip uninstall -y torch torchvision torchaudio
       pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu126
   ...then restart the kernel/session and re-run. cu126 wheels still
   ship sm_60 kernels; cu128+ wheels generally do not.

Research-readiness additions:
- Deterministic seeding across random/numpy/torch (+ CUDA determinism flags)
- Structured logging to both console and a run-specific log file
- A dataclass-based Config, overridable via CLI args or a YAML file, so
  every run is fully specified by one artifact you can archive
- A dataset integrity report (per split: images found, boxes found,
  classes actually present, empty-label images, corrupt/unreadable
  files) written to JSON before training starts
- Class-imbalance mitigation via inverse-frequency image oversampling
  (duplicated symlinks with re-suffixed filenames), since Ultralytics'
  detection trainer does not expose a native per-class weighted sampler
- Stronger, imbalance-aware augmentation defaults (higher HSV, mosaic,
  mixup, copy-paste, multi-scale) tuned for small, texture-dependent
  pest objects
- A held-out reproducibility manifest (package versions, git commit if
  available, full resolved config) saved alongside the run
- Per-class AP / precision / recall exported to CSV (not just aggregate
  mAP), so long-tail classes can be inspected individually
- Test-time augmentation (TTA) enabled for final evaluation and export
- A results markdown summary auto-generated at the end of the run,
  written in a format suitable for pasting into a paper/report
- A real GPU-kernel preflight check (not just cuda.is_available()) that
  catches sm_60/sm_50-vs-modern-wheel mismatches (e.g. Tesla P100) before
  training starts, with an actionable fix message instead of a crash
  mid-`model.to(device)`.

USAGE
-----
    python ip102_yolo_detection_research.py --config config.yaml
    # or override individual fields:
    python ip102_yolo_detection_research.py --model yolov8m.pt --epochs 100 --imgsz 640
=============================================================================
"""

from __future__ import annotations

import argparse
import dataclasses
import json
import logging
import os
import random
import subprocess
import sys
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from dataclasses import dataclass, field, asdict
from pathlib import Path
from typing import Optional

import numpy as np
import yaml

# ---------------------------------------------------------------------------
# Dependency bootstrap (kept minimal and explicit rather than silently
# pip-installing on every run — research code should make its environment
# assumptions visible, not paper over them).
# ---------------------------------------------------------------------------
try:
    import torch
    from ultralytics import YOLO
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "ultralytics"],
        check=True,
    )
    import torch
    from ultralytics import YOLO


# =============================================================================
# Config
# =============================================================================

@dataclass
class Config:
    # --- Paths -----------------------------------------------------------
    # Default matches the attached Kaggle dataset layout, where
    # Annotations/ and JPEGImages/ sit directly under the dataset root
    # (no nested "Detection" subfolder, and no ImageSets/Main split files —
    # see auto_generate_splits below).
    ip102_root: str = "/kaggle/input/datasets/parserpixy/ip102-dataset"
    work_dir: str = "/kaggle/working/ip102_yolo"
    output_dir: str = "/kaggle/working/runs"
    run_name: str = "ip102_yolo_research"
    # Extensions to try when locating an image for a given annotation stem;
    # tried in order, first match wins.
    image_extensions: tuple = (".jpg", ".jpeg", ".JPG", ".JPEG", ".png", ".PNG")

    # --- Dataset expectations ---------------------------------------------
    # None = auto-detect: the class count is taken from whatever is
    # actually discovered by scanning classes.txt (if present) or the XML
    # annotations (if not), and that discovered count becomes the target
    # every split is checked against. Set an explicit int only if you
    # already know the canonical class count for your specific dataset
    # variant (the official IP102 classification set is 102 classes, but
    # the public detection subset commonly ships with ~97 classes/~19k
    # annotated images — the two numbers are NOT interchangeable).
    expected_num_classes: Optional[int] = None
    # If False, the run aborts when the resolved class count != expected.
    # If True, it proceeds but the mismatch is logged prominently and
    # recorded in the integrity report — use only if you've deliberately
    # scoped down the class set.
    allow_class_count_mismatch: bool = False
    splits: tuple = ("train", "val", "test")

    # --- Split generation ---------------------------------------------------
    # This dataset variant ships Annotations/ + JPEGImages/ only, with no
    # ImageSets/Main/{train,val,test}.txt. If those official split files
    # are found, they're used as-is (reproduces the paper's exact split).
    # Otherwise, when auto_generate_splits is True, a stratified split is
    # generated from every annotated image and cached under
    # work_dir/ImageSets/Main/ so re-runs are stable across sessions.
    auto_generate_splits: bool = True
    train_fraction: float = 0.8
    val_fraction: float = 0.1
    test_fraction: float = 0.1

    # --- Class imbalance handling -------------------------------------------
    # Oversamples images containing rare classes so that, in expectation,
    # every class is seen a comparable number of times per epoch. This is
    # a data-level fix (Ultralytics has no built-in class-weighted sampler
    # for detection). Set to 1.0 to disable (no oversampling).
    enable_class_balancing: bool = True
    # Caps how many times any single image can be duplicated, to avoid
    # extreme oversampling of a handful of images from ultra-rare classes
    # collapsing the effective dataset diversity.
    max_oversample_factor: int = 8

    # --- Model / training ---------------------------------------------------
    model_weights: str = "yolov8m.pt"   # 'm' over 's': more capacity for
                                          # fine-grained inter-class confusion
    epochs: int = 30                    # reduced from 150 — with cos_lr +
                                          # patience-based early stopping,
                                          # 30 is enough to see whether the
                                          # run is converging without
                                          # burning a full Kaggle session on
                                          # a first pass. Raise this back up
                                          # once you've confirmed the
                                          # pipeline/data/hyperparams are
                                          # correct and want a final run.
    imgsz: int = 512                    # reduced from 640 — roughly halves
                                          # per-image compute (640^2 vs
                                          # 512^2 ≈ 1.56x fewer pixels) at
                                          # some cost to small-object recall.
                                          # Bump back to 640 for a final run
                                          # if pests are small in your images.
    batch: int = 16
    patience: int = 10                  # reduced from 30 — stops sooner
                                          # once val mAP plateaus, which
                                          # matters more now that epochs is
                                          # also lower
    optimizer: str = "auto"
    lr0: float = 0.01
    lrf: float = 0.01
    cos_lr: bool = True
    warmup_epochs: float = 3.0          # reduced from 5.0 to match the
                                          # shorter total epoch budget —
                                          # warmup shouldn't eat a sixth of
                                          # a 30-epoch run
    weight_decay: float = 0.0005
    dropout: float = 0.0
    workers: int = 2
    device: int | str = 0
    seed: int = 42
    amp: bool = True
    multi_scale: bool = False           # disabled by default for speed —
                                          # multi-scale training runs each
                                          # batch through the network at a
                                          # randomly resized resolution,
                                          # which improves scale robustness
                                          # but adds real per-batch overhead.
                                          # Re-enable for a final accuracy-
                                          # focused run.
    cache: str = "ram"                  # caches all images (post-resize)
                                          # in RAM after the first epoch, so
                                          # epochs 2+ skip disk I/O and JPEG
                                          # decoding entirely — usually the
                                          # single biggest speed win
                                          # Ultralytics offers. Set to
                                          # "disk" instead if the dataset is
                                          # too large for available RAM, or
                                          # False to disable.

    # --- Augmentation (raised vs. baseline defaults; pests are small,
    #     texture/color-dependent, and photographed under variable field
    #     lighting, per the Swin-AARNet paper's discussion of complex
    #     backgrounds and lighting variation) -----------------------------
    hsv_h: float = 0.02
    hsv_s: float = 0.8
    hsv_v: float = 0.5
    degrees: float = 10.0
    translate: float = 0.15
    scale: float = 0.6
    shear: float = 2.0
    flipud: float = 0.1
    fliplr: float = 0.5
    mosaic: float = 1.0
    mixup: float = 0.15
    copy_paste: float = 0.1
    close_mosaic: int = 5               # reduced from 15 — proportional to
                                          # the shorter 30-epoch run; disabling
                                          # mosaic for the final ~1/6 of
                                          # training (not literally the last
                                          # 15 of only 30 epochs) still gives
                                          # the model some "clean" epochs
                                          # before validation

    # --- Evaluation / export ------------------------------------------------
    tta_predict: bool = True
    conf_thres: float = 0.25
    export_formats: tuple = ("onnx", "torchscript")

    @classmethod
    def from_yaml(cls, path: str) -> "Config":
        with open(path, "r") as f:
            raw = yaml.safe_load(f) or {}
        return cls(**raw)

    def to_yaml(self, path: str) -> None:
        with open(path, "w") as f:
            yaml.safe_dump(asdict(self), f, sort_keys=False)


def _running_in_notebook() -> bool:
    """Detects Jupyter/Kaggle/Colab kernels, which inject their own launcher
    args (e.g. '-f /root/.../kernel-....json') into sys.argv. argparse must
    not try to strictly parse those, or it raises 'unrecognized arguments'
    exactly like the error seen when this script is run/imported in a
    Kaggle notebook cell instead of from a terminal.
    """
    try:
        from IPython import get_ipython
        return get_ipython() is not None
    except Exception:
        return False


def parse_args(argv: Optional[list[str]] = None) -> Config:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--config", type=str, default=None,
                         help="Path to a YAML config file. CLI flags below "
                              "override values loaded from it.")
    # Expose the most commonly-tuned fields directly on the CLI; anything
    # else can go in the YAML file.
    for f in dataclasses.fields(Config):
        if f.name == "splits" or f.name == "export_formats":
            continue  # tuple fields: YAML-only, not worth CLI plumbing
        parser.add_argument(f"--{f.name}", type=type(f.default) if f.default is not None else str,
                             default=None, help=f"Override Config.{f.name}")

    # In a notebook, sys.argv contains Jupyter's own launcher flags (e.g.
    # '-f kernel-....json'), not flags meant for this script. Use
    # parse_known_args() and silently ignore anything unrecognized instead
    # of erroring out. From a real terminal, parse_args() as normal so
    # genuine typos in flags still surface as errors.
    if argv is not None:
        args, unknown = parser.parse_known_args(argv)
    elif _running_in_notebook():
        args, unknown = parser.parse_known_args([])  # ignore sys.argv entirely
        if unknown:
            logging.getLogger("ip102_research").debug(f"Ignored notebook launcher args: {unknown}")
    else:
        args = parser.parse_args()

    cfg = Config.from_yaml(args.config) if args.config else Config()
    overrides = {k: v for k, v in vars(args).items() if k != "config" and v is not None}
    cfg = dataclasses.replace(cfg, **overrides)
    return cfg


# =============================================================================
# Reproducibility
# =============================================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    # Deterministic cuDNN costs some throughput but is worth it for a
    # research artifact where reruns should be comparable.
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)


def write_reproducibility_manifest(cfg: Config, out_path: Path) -> None:
    manifest = {
        "config": asdict(cfg),
        "python_version": sys.version,
        "torch_version": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
        "ultralytics_version": _safe_get_ultralytics_version(),
        "git_commit": _safe_get_git_commit(),
    }
    with open(out_path, "w") as f:
        json.dump(manifest, f, indent=2)


def _safe_get_ultralytics_version() -> Optional[str]:
    try:
        import ultralytics
        return ultralytics.__version__
    except Exception:
        return None


def _safe_get_git_commit() -> Optional[str]:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"], stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        return None


# =============================================================================
# Logging
# =============================================================================

def setup_logging(log_path: Path) -> logging.Logger:
    log_path.parent.mkdir(parents=True, exist_ok=True)
    logger = logging.getLogger("ip102_research")
    logger.setLevel(logging.INFO)
    logger.handlers.clear()

    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s",
                             datefmt="%Y-%m-%d %H:%M:%S")

    console = logging.StreamHandler(sys.stdout)
    console.setFormatter(fmt)
    logger.addHandler(console)

    file_handler = logging.FileHandler(log_path)
    file_handler.setFormatter(fmt)
    logger.addHandler(file_handler)

    return logger


# =============================================================================
# GPU / CUDA kernel preflight
# =============================================================================

def check_cuda_kernel_compatibility(logger: logging.Logger) -> tuple[bool, Optional[str]]:
    """Verifies the visible CUDA device can actually EXECUTE kernels, not
    just that it is detected. torch.cuda.is_available() only checks device
    discovery — it returns True even when the installed PyTorch wheel has
    no compiled kernels for this GPU's compute capability, which is exactly
    what happens on a Tesla P100 (sm_60) with a modern PyTorch build whose
    CUDA wheel index dropped sm_50/sm_60 support (common on cu128+ wheels;
    cu126 and earlier still ship sm_60). Left undetected, that mismatch
    surfaces much later and much less clearly, as an AcceleratorError deep
    inside ultralytics' `model.to(device)` call mid-training.

    Returns (ok, failure_reason). failure_reason is one of:
      None                  -> ok, GPU kernels work
      "cuda_unavailable"    -> no CUDA device visible at all
      "kernel_incompatible" -> device visible, but no usable kernels for it
    """
    if not torch.cuda.is_available():
        return False, "cuda_unavailable"

    try:
        probe = torch.randn(8, 8, device="cuda")
        _ = probe @ probe
        torch.cuda.synchronize()
        return True, None
    except RuntimeError as e:
        msg = str(e)
        if "no kernel image" in msg.lower():
            try:
                cap = torch.cuda.get_device_capability(0)
                cap_str = f"sm_{cap[0]}{cap[1]}"
            except Exception:
                cap_str = "unknown"
            try:
                gpu_name = torch.cuda.get_device_name(0)
            except Exception:
                gpu_name = "unknown GPU"
            logger.error(
                f"GPU '{gpu_name}' (compute capability {cap_str}) is visible to "
                f"PyTorch, but this PyTorch build ({torch.__version__}, CUDA "
                f"{torch.version.cuda}) has no compiled kernels for it. This is "
                f"the classic Tesla P100 (sm_60) issue: recent PyTorch CUDA "
                f"wheel indices (cu128 and newer) dropped sm_50/sm_60 support "
                f"in favor of newer architectures, so torch.cuda.is_available() "
                f"reports True but every real GPU op fails.\n\n"
                f"FIX (must be run BEFORE this script — torch cannot be "
                f"hot-swapped once already imported in this process):\n"
                f"    pip uninstall -y torch torchvision torchaudio\n"
                f"    pip install torch torchvision torchaudio "
                f"--index-url https://download.pytorch.org/whl/cu126\n"
                f"Then RESTART the kernel/session and re-run this script.\n"
                f"(If cu126 still doesn't work, try cu121 next — both predate "
                f"the sm_60 drop.)"
            )
            return False, "kernel_incompatible"
        # Some other CUDA runtime error we don't specifically handle here —
        # surface it as-is rather than mislabeling it.
        raise


# =============================================================================
# Dataset: class list, VOC->YOLO conversion, integrity validation
# =============================================================================

def load_or_build_class_list(annotations_dir: Path, classes_file: Path,
                              logger: logging.Logger) -> list[str]:
    if classes_file.exists():
        with open(classes_file, "r") as f:
            names = [line.strip() for line in f if line.strip()]
        logger.info(f"Loaded {len(names)} classes from {classes_file}")
        return names

    logger.warning(
        "classes.txt not found — scanning XML annotations to infer class "
        "names (alphabetical order). This order will NOT match official "
        "IP102 class ids and results won't be comparable to published "
        "benchmarks unless you supply the canonical classes.txt."
    )
    names = set()
    for xml_path in annotations_dir.glob("*.xml"):
        try:
            tree = ET.parse(xml_path)
            for obj in tree.getroot().findall("object"):
                names.add(obj.find("name").text.strip())
        except ET.ParseError as e:
            logger.warning(f"Skipping malformed XML during class scan: {xml_path.name} ({e})")
    names = sorted(names)
    logger.info(f"Discovered {len(names)} classes from annotations.")
    return names


def convert_voc_to_yolo(xml_path: Path, class_to_id: dict,
                         logger: logging.Logger) -> Optional[list[str]]:
    """Parses one Pascal VOC XML file into YOLO-format label lines.
    Returns None (caller skips the image) if the XML is malformed, has
    invalid dimensions, or yields zero valid boxes after clipping.
    Every skip reason is logged, not silently dropped.
    """
    try:
        tree = ET.parse(xml_path)
    except ET.ParseError as e:
        logger.debug(f"[skip] {xml_path.name}: XML parse error ({e})")
        return None

    root = tree.getroot()
    size = root.find("size")
    if size is None:
        logger.debug(f"[skip] {xml_path.name}: missing <size> tag")
        return None

    try:
        img_w = int(size.find("width").text)
        img_h = int(size.find("height").text)
    except (AttributeError, TypeError, ValueError):
        logger.debug(f"[skip] {xml_path.name}: unparseable width/height")
        return None

    if img_w <= 0 or img_h <= 0:
        logger.debug(f"[skip] {xml_path.name}: non-positive dimensions ({img_w}x{img_h})")
        return None

    lines = []
    for obj in root.findall("object"):
        name_el = obj.find("name")
        if name_el is None or not name_el.text:
            continue
        name = name_el.text.strip()
        if name not in class_to_id:
            logger.debug(f"[skip-object] {xml_path.name}: unmapped class '{name}'")
            continue
        class_id = class_to_id[name]

        bbox = obj.find("bndbox")
        if bbox is None:
            continue
        try:
            xmin = float(bbox.find("xmin").text)
            ymin = float(bbox.find("ymin").text)
            xmax = float(bbox.find("xmax").text)
            ymax = float(bbox.find("ymax").text)
        except (AttributeError, TypeError, ValueError):
            continue

        xmin, xmax = max(0, xmin), min(img_w, xmax)
        ymin, ymax = max(0, ymin), min(img_h, ymax)
        if xmax <= xmin or ymax <= ymin:
            continue

        x_center = ((xmin + xmax) / 2) / img_w
        y_center = ((ymin + ymax) / 2) / img_h
        width = (xmax - xmin) / img_w
        height = (ymax - ymin) / img_h
        lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")

    return lines if lines else None


def find_image_path(images_dir: Path, stem: str, extensions: tuple) -> Optional[Path]:
    """Locates the image file for a given annotation stem, trying several
    extensions since not every IP102 mirror uses a consistent lowercase
    '.jpg' — some re-uploads mix in .JPG/.png files.
    """
    for ext in extensions:
        candidate = images_dir / f"{stem}{ext}"
        if candidate.exists():
            return candidate
    return None


def discover_annotated_ids(images_dir: Path, annotations_dir: Path,
                            image_extensions: tuple, logger: logging.Logger) -> list[str]:
    """Scans every XML annotation and keeps the ids that have a matching
    image on disk. This is the ground truth list of usable samples when no
    official ImageSets/Main split files exist.
    """
    ids, missing_image = [], 0
    for xml_path in sorted(annotations_dir.glob("*.xml")):
        stem = xml_path.stem
        if find_image_path(images_dir, stem, image_extensions) is not None:
            ids.append(stem)
        else:
            missing_image += 1
    logger.info(f"Discovered {len(ids)} annotated images with a matching image file "
                f"({missing_image} annotations had no matching image and were excluded).")
    return ids


def get_classes_per_image(annotations_dir: Path, ids: list[str], class_to_id: dict,
                           logger: logging.Logger) -> dict:
    """Lightweight pre-parse (class names only, no box math) used purely to
    drive stratified split generation — lets the split-builder know which
    classes each image touches before doing the full VOC->YOLO conversion.
    """
    classes_per_image = {}
    for image_id in ids:
        xml_path = annotations_dir / f"{image_id}.xml"
        try:
            tree = ET.parse(xml_path)
        except ET.ParseError:
            classes_per_image[image_id] = set()
            continue
        present = set()
        for obj in tree.getroot().findall("object"):
            name_el = obj.find("name")
            if name_el is not None and name_el.text and name_el.text.strip() in class_to_id:
                present.add(class_to_id[name_el.text.strip()])
        classes_per_image[image_id] = present
    return classes_per_image


def create_stratified_splits(ids: list[str], classes_per_image: dict, cfg: Config,
                              logger: logging.Logger) -> dict:
    """Random split by fractions, followed by a rescue pass that guarantees
    every class present in the dataset has at least one example in the
    train split (a plain random split can otherwise strand a rare class
    entirely in val/test, especially given IP102's long-tailed
    distribution — imbalance ratios exceeding 91:1 have been reported on
    the classification set, and the detection subset is smaller still).
    """
    total_frac = cfg.train_fraction + cfg.val_fraction + cfg.test_fraction
    if abs(total_frac - 1.0) > 1e-6:
        raise ValueError(
            f"train_fraction + val_fraction + test_fraction must sum to 1.0, "
            f"got {total_frac} ({cfg.train_fraction}, {cfg.val_fraction}, {cfg.test_fraction})"
        )

    rng = random.Random(cfg.seed)
    shuffled = ids.copy()
    rng.shuffle(shuffled)

    n = len(shuffled)
    n_train = int(round(n * cfg.train_fraction))
    n_val = int(round(n * cfg.val_fraction))

    train_ids = shuffled[:n_train]
    val_ids = shuffled[n_train:n_train + n_val]
    test_ids = shuffled[n_train + n_val:]

    all_classes = set()
    for s in classes_per_image.values():
        all_classes |= s

    train_classes = set()
    for i in train_ids:
        train_classes |= classes_per_image.get(i, set())
    missing_from_train = sorted(all_classes - train_classes)

    if missing_from_train:
        logger.warning(
            f"{len(missing_from_train)} class(es) absent from the randomly-split "
            f"train set — rescuing by moving one donor image per missing class "
            f"from val/test into train."
        )
        for cls_id in missing_from_train:
            donor_pool_name, donor_id = None, None
            for pool_name, pool in (("val", val_ids), ("test", test_ids)):
                for image_id in pool:
                    if cls_id in classes_per_image.get(image_id, set()):
                        donor_pool_name, donor_id = pool_name, image_id
                        break
                if donor_id:
                    break
            if donor_id is None:
                logger.error(f"class_id {cls_id} has zero annotated images anywhere in "
                              f"the dataset — cannot rescue into train. This class will "
                              f"be entirely absent and should show up in the integrity report.")
                continue
            (val_ids if donor_pool_name == "val" else test_ids).remove(donor_id)
            train_ids.append(donor_id)

    logger.info(f"Auto-generated splits: train={len(train_ids)} val={len(val_ids)} "
                f"test={len(test_ids)} (seed={cfg.seed})")
    return {"train": train_ids, "val": val_ids, "test": test_ids}


def write_split_files(splits_ids: dict, imagesets_dir: Path, logger: logging.Logger) -> None:
    imagesets_dir.mkdir(parents=True, exist_ok=True)
    for split, ids in splits_ids.items():
        split_file = imagesets_dir / f"{split}.txt"
        with open(split_file, "w") as f:
            f.write("\n".join(ids))
        logger.info(f"Wrote {len(ids)} ids to {split_file}")


def resolve_splits(cfg: Config, images_dir: Path, annotations_dir: Path,
                    class_to_id: dict, work_dir: Path, logger: logging.Logger) -> Path:
    """Returns the ImageSets/Main directory to read splits from. Uses the
    dataset's own official split files if present; otherwise generates a
    stratified split and caches it under work_dir so repeated runs (and
    the class-balancing / integrity steps downstream) are stable.
    """
    official_dir = Path(cfg.ip102_root) / "ImageSets" / "Main"
    has_official = all((official_dir / f"{s}.txt").exists() for s in cfg.splits)

    if has_official:
        logger.info(f"Using official split files found at {official_dir}")
        return official_dir

    if not cfg.auto_generate_splits:
        raise RuntimeError(
            f"No official ImageSets/Main split files found at {official_dir}, and "
            f"auto_generate_splits is False. Either set auto_generate_splits=True "
            f"or point ip102_root at a dataset that ships official splits."
        )

    logger.warning(
        f"No official ImageSets/Main split files found at {official_dir}. "
        f"Auto-generating a {cfg.train_fraction:.0%}/{cfg.val_fraction:.0%}/"
        f"{cfg.test_fraction:.0%} train/val/test split from all annotated images "
        f"(seed={cfg.seed}, cached under {work_dir}/ImageSets/Main so this is "
        f"reproducible across reruns)."
    )
    ids = discover_annotated_ids(images_dir, annotations_dir, cfg.image_extensions, logger)
    if not ids:
        raise RuntimeError(
            f"No annotated images with a matching image file were found under "
            f"{images_dir} / {annotations_dir}. Check that JPEGImages/ and "
            f"Annotations/ actually contain matching files (same stem, e.g. "
            f"'00001.jpg' + '00001.xml')."
        )
    classes_per_image = get_classes_per_image(annotations_dir, ids, class_to_id, logger)
    splits_ids = create_stratified_splits(ids, classes_per_image, cfg, logger)

    generated_dir = work_dir / "ImageSets" / "Main"
    write_split_files(splits_ids, generated_dir, logger)
    return generated_dir


def read_split_ids(imagesets_dir: Path, split: str, logger: logging.Logger) -> list[str]:
    split_file = imagesets_dir / f"{split}.txt"
    if not split_file.exists():
        logger.warning(f"[{split}] split file not found: {split_file}")
        return []

    ids, malformed = [], 0
    with open(split_file, "r") as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            tokens = line.split()
            if not tokens:
                malformed += 1
                continue
            ids.append(tokens[0])
    if malformed:
        logger.warning(f"[{split}] {malformed} malformed lines in {split_file.name}")
    return ids


def build_yolo_split(split: str, cfg: Config, class_to_id: dict,
                      images_dir: Path, annotations_dir: Path,
                      imagesets_dir: Path, yolo_images_dir: Path,
                      yolo_labels_dir: Path, logger: logging.Logger) -> dict:
    """Builds one split and returns a per-split integrity report dict."""
    report = {
        "split": split,
        "ids_in_split_file": 0,
        "missing_image": 0,
        "missing_annotation": 0,
        "no_valid_boxes": 0,
        "written": 0,
        "class_counts": Counter(),
    }

    image_ids = read_split_ids(imagesets_dir, split, logger)
    report["ids_in_split_file"] = len(image_ids)
    if not image_ids:
        logger.warning(f"[{split}] no ids found — skipping split entirely.")
        return report

    for image_id in image_ids:
        src_img = find_image_path(images_dir, image_id, cfg.image_extensions)
        src_xml = annotations_dir / f"{image_id}.xml"

        if src_img is None:
            report["missing_image"] += 1
            continue
        if not src_xml.exists():
            report["missing_annotation"] += 1
            continue

        yolo_lines = convert_voc_to_yolo(src_xml, class_to_id, logger)
        if not yolo_lines:
            report["no_valid_boxes"] += 1
            continue

        dst_img = yolo_images_dir / split / src_img.name
        dst_label = yolo_labels_dir / split / f"{image_id}.txt"

        if not dst_img.exists():
            os.symlink(src_img.resolve(), dst_img)
        with open(dst_label, "w") as f:
            f.write("\n".join(yolo_lines))

        for line in yolo_lines:
            cls_id = int(line.split()[0])
            report["class_counts"][cls_id] += 1

        report["written"] += 1

    logger.info(
        f"[{split}] ids={report['ids_in_split_file']} written={report['written']} "
        f"missing_image={report['missing_image']} missing_ann={report['missing_annotation']} "
        f"no_valid_boxes={report['no_valid_boxes']} "
        f"classes_present={len(report['class_counts'])}"
    )
    return report


def validate_dataset_integrity(reports: dict, cfg: Config, logger: logging.Logger,
                                out_path: Path) -> None:
    """This is the check that would have caught the '101 vs 102 classes'
    issue immediately and loudly instead of leaving it to be spotted later
    in a screenshot. Writes a full JSON report and raises if the resolved
    class count doesn't match expectations (unless explicitly overridden).
    """
    all_classes_present = set()
    for r in reports.values():
        all_classes_present |= set(r["class_counts"].keys())

    n_present = len(all_classes_present)
    missing_classes = sorted(set(range(cfg.expected_num_classes)) - all_classes_present)

    summary = {
        "expected_num_classes": cfg.expected_num_classes,
        "num_classes_present_across_all_splits": n_present,
        "missing_class_ids": missing_classes,
        "per_split": {
            split: {
                "written": r["written"],
                "missing_image": r["missing_image"],
                "missing_annotation": r["missing_annotation"],
                "no_valid_boxes": r["no_valid_boxes"],
                "num_classes_present": len(r["class_counts"]),
                "class_counts": dict(sorted(r["class_counts"].items())),
            }
            for split, r in reports.items()
        },
    }

    with open(out_path, "w") as f:
        json.dump(summary, f, indent=2)
    logger.info(f"Dataset integrity report written to {out_path}")

    if missing_classes:
        msg = (
            f"Expected {cfg.expected_num_classes} classes but only "
            f"{n_present} are present across all splits after conversion. "
            f"Missing class ids: {missing_classes}. "
            f"Check the class-name mapping in classes.txt against the "
            f"actual <name> values used in the XML annotations — a single "
            f"naming mismatch (e.g. trailing whitespace, case difference) "
            f"is the most common cause."
        )
        if cfg.allow_class_count_mismatch:
            logger.warning("MISMATCH ALLOWED BY CONFIG — proceeding anyway. " + msg)
        else:
            logger.error(msg)
            raise RuntimeError(msg)

    logger.info(f"Dataset integrity OK: all {cfg.expected_num_classes} classes present.")


# =============================================================================
# Class-imbalance mitigation: inverse-frequency oversampling
# =============================================================================

def apply_class_balancing(split: str, report: dict, cfg: Config,
                           yolo_images_dir: Path, yolo_labels_dir: Path,
                           logger: logging.Logger) -> None:
    """Duplicates (via symlink) images containing under-represented classes
    so that, in expectation, each class contributes a more comparable
    number of instances per epoch. Only applied to the training split —
    val/test must remain untouched to keep evaluation numbers honest.
    """
    if split != "train" or not cfg.enable_class_balancing:
        return

    class_counts = report["class_counts"]
    if not class_counts:
        return

    max_count = max(class_counts.values())
    label_dir = yolo_labels_dir / split
    image_dir = yolo_images_dir / split

    # Compute an oversample factor per image based on the rarest class it
    # contains (rescuing the rarest class in a multi-object image gives the
    # biggest imbalance benefit per duplicate).
    n_duplicated = 0
    for label_file in sorted(label_dir.glob("*.txt")):
        with open(label_file, "r") as f:
            class_ids_in_image = {int(line.split()[0]) for line in f if line.strip()}
        if not class_ids_in_image:
            continue

        rarest_count = min(class_counts[c] for c in class_ids_in_image)
        factor = min(cfg.max_oversample_factor, max(1, round(max_count / max(1, rarest_count))))
        if factor <= 1:
            continue

        stem = label_file.stem
        src_img = None
        for ext in (".jpg", ".jpeg", ".png"):
            candidate = image_dir / f"{stem}{ext}"
            if candidate.exists():
                src_img = candidate
                break
        if src_img is None:
            continue

        for dup_idx in range(1, factor):
            dup_img = image_dir / f"{stem}__dup{dup_idx}{src_img.suffix}"
            dup_label = label_dir / f"{stem}__dup{dup_idx}.txt"
            if not dup_img.exists():
                os.symlink(src_img.resolve(), dup_img)
            if not dup_label.exists():
                dup_label.write_text(label_file.read_text())
            n_duplicated += 1

    logger.info(
        f"[{split}] class-balancing oversampling: created {n_duplicated} duplicate "
        f"image/label pairs (max_oversample_factor={cfg.max_oversample_factor})."
    )


# =============================================================================
# Main pipeline
# =============================================================================

def main(config_overrides: Optional[dict] = None) -> None:
    """Entry point. In a terminal, reads CLI flags/--config as usual.
    In a notebook, call main({"epochs": 100, "model_weights": "yolov8m.pt", ...})
    to override fields directly in Python — this sidesteps argparse and
    sys.argv entirely, which is the cleanest way to run this from a Kaggle
    cell (no 'unrecognized arguments' issue possible this way).
    """
    cfg = parse_args()
    if config_overrides:
        cfg = dataclasses.replace(cfg, **config_overrides)
    seed_everything(cfg.seed)

    work_dir = Path(cfg.work_dir)
    yolo_images_dir = work_dir / "images"
    yolo_labels_dir = work_dir / "labels"
    run_dir = Path(cfg.output_dir) / cfg.run_name
    run_dir.mkdir(parents=True, exist_ok=True)

    logger = setup_logging(run_dir / "pipeline.log")
    logger.info("=" * 79)
    logger.info("IP102 Detection — research-ready pipeline starting")
    logger.info("=" * 79)

    # --- GPU kernel preflight (do this FIRST, before any dataset work, so ---
    # --- a broken CUDA/torch install fails in seconds, not after minutes ---
    # --- of dataset conversion) ----------------------------------------------
    requested_gpu = isinstance(cfg.device, int) or (
        isinstance(cfg.device, str) and cfg.device.replace(",", "").isdigit()
    )
    if requested_gpu:
        gpu_ok, failure_reason = check_cuda_kernel_compatibility(logger)
        if not gpu_ok:
            if failure_reason == "cuda_unavailable":
                logger.warning(
                    f"cfg.device={cfg.device!r} requests a GPU, but "
                    f"torch.cuda.is_available() is False (device_count="
                    f"{torch.cuda.device_count()}). This almost always means the "
                    f"Kaggle notebook's Accelerator is set to 'None', or the "
                    f"weekly GPU quota has been exhausted — check "
                    f"Settings > Accelerator in the Kaggle sidebar and restart "
                    f"the session with a GPU attached, then re-run. "
                    f"Falling back to device='cpu' for now so this run doesn't "
                    f"crash, but training {cfg.model_weights} for {cfg.epochs} "
                    f"epochs at imgsz={cfg.imgsz} on CPU will be extremely slow "
                    f"(likely hours-to-days rather than the ~minutes/epoch a T4 "
                    f"gives you) — strongly recommend fixing the accelerator and "
                    f"re-running rather than letting this CPU run finish."
                )
                cfg = dataclasses.replace(cfg, device="cpu")
            elif failure_reason == "kernel_incompatible":
                # Detailed diagnosis + exact fix command already logged inside
                # check_cuda_kernel_compatibility(). Falling back to CPU here
                # would silently mask the real fix needed and would be
                # impractical anyway (YOLOv8m for cfg.epochs epochs at
                # cfg.imgsz on CPU is not a realistic option) — so this stops
                # the run instead of limping along.
                raise RuntimeError(
                    "GPU is visible but cannot execute PyTorch kernels (see "
                    "the error logged above for the exact fix and reinstall "
                    "command). Reinstall PyTorch with a compatible CUDA wheel "
                    "index, RESTART the kernel/session, then re-run this "
                    "script. Not falling back to CPU: training at this "
                    "epoch/imgsz budget on CPU would be impractically slow "
                    "and would hide the real problem."
                )

    cfg.to_yaml(str(run_dir / "resolved_config.yaml"))
    write_reproducibility_manifest(cfg, run_dir / "reproducibility_manifest.json")
    logger.info(f"Resolved config and reproducibility manifest saved to {run_dir}")

    ip102_root = Path(cfg.ip102_root)
    images_dir = ip102_root / "JPEGImages"
    annotations_dir = ip102_root / "Annotations"
    classes_file = ip102_root / "classes.txt"

    for split in cfg.splits:
        (yolo_images_dir / split).mkdir(parents=True, exist_ok=True)
        (yolo_labels_dir / split).mkdir(parents=True, exist_ok=True)

    # --- Class list & conversion ------------------------------------------
    class_names = load_or_build_class_list(annotations_dir, classes_file, logger)
    if not class_names:
        raise RuntimeError(
            f"No classes found. Checked:\n"
            f"  classes_file:     {classes_file}  (exists={classes_file.exists()})\n"
            f"  annotations_dir:  {annotations_dir}  (exists={annotations_dir.exists()}, "
            f"xml_count={len(list(annotations_dir.glob('*.xml'))) if annotations_dir.exists() else 'N/A'})\n"
            f"  ip102_root:       {ip102_root}  (exists={ip102_root.exists()})\n"
            f"This usually means ip102_root is pointing at the wrong dataset/layout. "
            f"Run `!find /kaggle/input -maxdepth 4 -type d` in a cell to see what's "
            f"actually attached, then set cfg['ip102_root'] to the correct path. Note "
            f"this script expects a VOC-style detection layout (JPEGImages/, "
            f"Annotations/) — a classification-layout dataset (classes.txt + "
            f"train.txt/val.txt/test.txt at the top level, no XML annotations) will "
            f"always fail this check, since there are no bounding boxes to convert."
        )
    class_to_id = {name: i for i, name in enumerate(class_names)}

    # Resolve the class-count target for the integrity check. auto-detect
    # (the default) uses whatever was actually discovered above, so the
    # check validates internal consistency (did every discovered class
    # survive into the splits?) rather than assuming a hardcoded number
    # that may not match this particular dataset variant.
    if cfg.expected_num_classes is None:
        cfg = dataclasses.replace(cfg, expected_num_classes=len(class_names))
        logger.info(f"expected_num_classes auto-detected as {cfg.expected_num_classes} "
                    f"(from classes.txt / annotation scan).")

    # --- Splits: official if present, otherwise auto-generated & cached ----
    imagesets_dir = resolve_splits(
        cfg, images_dir, annotations_dir, class_to_id, work_dir, logger
    )

    reports = {}
    for split in cfg.splits:
        reports[split] = build_yolo_split(
            split, cfg, class_to_id, images_dir, annotations_dir,
            imagesets_dir, yolo_images_dir, yolo_labels_dir, logger,
        )

    # --- Integrity gate (this is the check that catches the 101-vs-102 bug) --
    validate_dataset_integrity(
        reports, cfg, logger, run_dir / "dataset_integrity_report.json"
    )

    # --- Class balancing (train split only) --------------------------------
    for split in cfg.splits:
        apply_class_balancing(
            split, reports[split], cfg, yolo_images_dir, yolo_labels_dir, logger
        )

    # --- data.yaml -----------------------------------------------------------
    split_written = {s: reports[s]["written"] for s in cfg.splits}
    data_yaml_path = work_dir / "ip102.yaml"
    data_yaml = {
        "path": str(work_dir),
        "train": "images/train",
        "val": "images/val" if split_written.get("val", 0) > 0 else "images/train",
        "names": {i: name for i, name in enumerate(class_names)},
    }
    if split_written.get("test", 0) > 0:
        data_yaml["test"] = "images/test"

    with open(data_yaml_path, "w") as f:
        yaml.safe_dump(data_yaml, f, sort_keys=False)
    logger.info(f"Wrote data.yaml to {data_yaml_path}")

    # --- Train -----------------------------------------------------------------
    logger.info(f"Loading base weights: {cfg.model_weights}")
    model = YOLO(cfg.model_weights)

    logger.info("Starting training...")
    model.train(
        data=str(data_yaml_path),
        epochs=cfg.epochs,
        imgsz=cfg.imgsz,
        batch=cfg.batch,
        project=cfg.output_dir,
        name=cfg.run_name,
        patience=cfg.patience,
        device=cfg.device,
        workers=cfg.workers,
        optimizer=cfg.optimizer,
        lr0=cfg.lr0,
        lrf=cfg.lrf,
        cos_lr=cfg.cos_lr,
        warmup_epochs=cfg.warmup_epochs,
        weight_decay=cfg.weight_decay,
        dropout=cfg.dropout,
        amp=cfg.amp,
        multi_scale=cfg.multi_scale,
        cache=cfg.cache,
        hsv_h=cfg.hsv_h, hsv_s=cfg.hsv_s, hsv_v=cfg.hsv_v,
        degrees=cfg.degrees, translate=cfg.translate, scale=cfg.scale, shear=cfg.shear,
        flipud=cfg.flipud, fliplr=cfg.fliplr,
        mosaic=cfg.mosaic, mixup=cfg.mixup, copy_paste=cfg.copy_paste,
        close_mosaic=cfg.close_mosaic,
        seed=cfg.seed,
        deterministic=True,
        exist_ok=True,
        plots=True,
    )

    # --- Validate + per-class AP export --------------------------------------
    logger.info("Running validation...")
    metrics = model.val(data=str(data_yaml_path), split="val")
    logger.info(f"mAP50-95: {metrics.box.map:.4f}")
    logger.info(f"mAP50:    {metrics.box.map50:.4f}")
    logger.info(f"mAP75:    {metrics.box.map75:.4f}")

    per_class_path = run_dir / "per_class_ap.csv"
    with open(per_class_path, "w") as f:
        f.write("class_id,class_name,train_instances,AP50-95\n")
        maps = getattr(metrics.box, "maps", None)
        train_counts = reports.get("train", {}).get("class_counts", {})
        if maps is not None:
            for cls_id, ap in enumerate(maps):
                name = class_names[cls_id] if cls_id < len(class_names) else f"class_{cls_id}"
                f.write(f"{cls_id},{name},{train_counts.get(cls_id, 0)},{ap:.4f}\n")
    logger.info(f"Per-class AP written to {per_class_path} — inspect this for "
                f"long-tail classes with disproportionately low AP.")

    # --- Test-set inference (TTA) --------------------------------------------
    best_weights = Path(model.trainer.save_dir) / "weights" / "best.pt"
    inference_model = YOLO(str(best_weights))

    test_images_dir = yolo_images_dir / "test"
    if test_images_dir.exists() and any(test_images_dir.iterdir()):
        logger.info(f"Running inference on test split (TTA={cfg.tta_predict})...")
        preds = inference_model.predict(
            source=str(test_images_dir),
            imgsz=cfg.imgsz,
            conf=cfg.conf_thres,
            augment=cfg.tta_predict,
            save=True,
            project=cfg.output_dir,
            name=f"{cfg.run_name}_test_predictions",
            exist_ok=True,
        )
        logger.info(f"Ran inference on {len(preds)} test images.")
    else:
        logger.warning("No test split found — skipping inference step.")

    # --- Export ----------------------------------------------------------------
    for fmt in cfg.export_formats:
        logger.info(f"Exporting model to {fmt}...")
        inference_model.export(format=fmt, imgsz=cfg.imgsz)

    # --- Results summary (paper/report-ready) ------------------------------
    summary_path = run_dir / "RESULTS_SUMMARY.md"
    with open(summary_path, "w") as f:
        f.write(f"# IP102 Detection — Results Summary\n\n")
        f.write(f"**Run name:** {cfg.run_name}\n\n")
        f.write(f"**Model:** {cfg.model_weights} | **Epochs:** {cfg.epochs} | "
                f"**Image size:** {cfg.imgsz} | **Batch:** {cfg.batch}\n\n")
        f.write(f"**Class balancing:** {'enabled' if cfg.enable_class_balancing else 'disabled'} "
                f"(max oversample factor {cfg.max_oversample_factor})\n\n")
        f.write(f"## Validation metrics\n\n")
        f.write(f"| Metric | Value |\n|---|---|\n")
        f.write(f"| mAP50-95 | {metrics.box.map:.4f} |\n")
        f.write(f"| mAP50 | {metrics.box.map50:.4f} |\n")
        f.write(f"| mAP75 | {metrics.box.map75:.4f} |\n\n")
        f.write(f"Per-class AP: see `per_class_ap.csv`\n\n")
        f.write(f"Dataset integrity report: see `dataset_integrity_report.json`\n\n")
        f.write(f"Full resolved config: see `resolved_config.yaml`\n\n")
        f.write(f"Reproducibility manifest: see `reproducibility_manifest.json`\n")
    logger.info(f"Results summary written to {summary_path}")
    logger.info("Pipeline complete.")


if __name__ == "__main__":
    main()

2026-09-08 11:04:53 | INFO    | ===============================================================================
2026-09-08 11:04:53 | INFO    | IP102 Detection — research-ready pipeline starting
2026-09-08 11:04:53 | INFO    | ===============================================================================
2026-09-08 11:04:53 | INFO    | Resolved config and reproducibility manifest saved to /kaggle/working/runs/ip102_yolo_research
2026-09-08 11:04:53 | WARNING | classes.txt not found — scanning XML annotations to infer class names (alphabetical order). This order will NOT match official IP102 class ids and results won't be comparable to published benchmarks unless you supply the canonical classes.txt.
2026-09-08 11:08:30 | WARNING | Skipping malformed XML during class scan: IP087000986.xml (junk after document element: line 27, column 0)
2026-09-08 11:09:01 | INFO    | Discovered 97 classes from annotations.
2026-09-08 11:09:01 | INFO    | expected_num_classes auto-detected as 97 (from 

/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1333: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:185.)
  attn = ((q * self.scale).transpose(-2, -1) @ k).softmax(dim=-1)
/usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py:1334: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not de

AMP: checks passed ✅
WARNING ⚠️ train: Slow image access detected (ping: 0.0±0.0 ms, read: 3.2±2.7 MB/s, size: 24.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
train: Scanning /kaggle/working/ip102_yolo/labels/train... 83025 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 83025/83025 468.3it/s 2:570.0ss
train: New cache created: /kaggle/working/ip102_yolo/labels/train.cache
WARNING ⚠️ train: 91.2GB RAM required to cache images with 100% safety margin but only 28.6/31.3GB available, not caching images
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 3.6±2.7 MB/s, size: 30.2 KB). Use local storage instead of remote/mounted storage for better performance. See h

/usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py:397: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:185.)
  pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3).matmul(self.proj.type(pred_dist.dtype))


       1/30      6.08G      2.209      5.368      2.365         55        512: 0% ──────────── 5/5190 1.5it/s 3.5s<56:0113

...

       8/30      6.17G      1.222      1.549      1.416          2        512: 100% ━━━━━━━━━━━━ 5190/5190 2.0it/s 44:000.3ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 60/60 3.3it/s 18.1s0.3s
                   all       1898       2217      0.508      0.645      0.613      0.409

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


/usr/local/lib/python3.12/dist-packages/ultralytics/utils/loss.py:397: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:185.)
  pred_dist = pred_dist.view(b, a, 4, c // 4).softmax(3).matmul(self.proj.type(pred_dist.dtype))


       9/30      6.17G      1.344      1.805      1.511         47        512: 0% ──────────── 2/5190 1.1s/it 1.6s<1:32:05

/usr/local/lib/python3.12/dist-packages/ultralytics/optim/muon.py:44: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA >= 10.2. To enable deterministic behavior in this case, you must set an environment variable before running your PyTorch application: CUBLAS_WORKSPACE_CONFIG=:4096:8 or CUBLAS_WORKSPACE_CONFIG=:16:8. For more information, go to https://docs.nvidia.com/cuda/cublas/index.html#results-reproducibility (Triggered internally at ../aten/src/ATen/Context.cpp:185.)
  A = X @ X.transpose(-2, -1)
/usr/local/lib/python3.12/dist-packages/ultralytics/optim/muon.py:45: UserWarning: Deterministic behavior was enabled with either `torch.use_deterministic_algorithms(True)` or `at::Context::setDeterministicAlgorithms(true)`, but this operation is not deterministic because it uses CuBLAS and you have CUDA

       9/30      6.17G      1.202      1.477      1.399         54        512: 87% ━━━━━━━━━━╸─ 4547/5190 2.0it/s 38:45<5:272